# LeetCode 52 — N-Queens II: đếm số cách đặt N quân hậu (Hard)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def totalNQueens(self, n: int) -> int:

```

Bài #51 **N-Queens** yêu cầu *liệt kê* mọi bàn cờ. Bài #52 này chỉ hỏi: **có bao nhiêu lời giải?**
Nghe thì giống nhau, nhưng "chỉ cần đếm" mở ra nhiều cơ hội tối ưu: không cần dựng chuỗi `"..Q."`, không cần lưu danh sách lời giải, và có thể dồn toàn bộ trạng thái vào vài con số nguyên.

---

## Sau notebook này, bạn sẽ nắm được

1. Quân hậu tấn công theo những hướng nào, và cách mô tả **đường chéo** bằng `r - c` và `r + c`.
2. Cách brute force bằng **hoán vị** (permutation) và vì sao nó lãng phí.
3. **Backtracking** (quay lui) từ căn bản: chọn → đi sâu → bỏ chọn, và cách *đếm* bằng cách cộng kết quả của các nhánh con.
4. Ba mức tối ưu phép kiểm tra "ô này có an toàn không?": quét mảng `O(n)` → 3 `set` `O(1)` → **bitmask** (mặt nạ bit).
5. Các phép toán bit từ số 0: `<<`, `>>`, `&`, `|`, `~`, và mẹo `x & -x` lấy bit thấp nhất.
6. Mẹo **đối xứng gương**: chỉ thử nửa số cột ở hàng đầu rồi nhân 2 (cẩn thận với `n` lẻ).
7. Đo **số nút duyệt** để so sánh các cách một cách khách quan.

## 1. Đề bài

> The **n-queens** puzzle is the problem of placing `n` queens on an `n x n` chessboard such that no two queens attack each other.
>
> Given an integer `n`, return *the number of distinct solutions to the n-queens puzzle*.

Dịch: đặt `n` quân hậu lên bàn cờ `n x n` sao cho **không có hai quân hậu nào tấn công nhau**. Trả về **số lời giải khác nhau**.

**Example 1**

```text
Input: n = 4
Output: 2
Explanation: There are two distinct solutions to the 4-queens puzzle as shown.
```

```text
Lời giải A        Lời giải B
. Q . .           . . Q .
. . . Q           Q . . .
Q . . .           . . . Q
. . Q .           . Q . .
```

**Example 2**

```text
Input: n = 1
Output: 1
```

**Constraints**

- `1 <= n <= 9`

Chú ý: hai lời giải là "khác nhau" khi **tập ô đặt hậu khác nhau**. Lời giải B chính là lời giải A lật gương trái–phải, nhưng vẫn được đếm là 2 lời giải riêng.

## 2. Quân hậu tấn công thế nào?

Quân hậu (queen) đi được bao xa tùy ý theo **8 hướng**: ngang, dọc và 2 đường chéo. Hai hậu tấn công nhau nếu chúng:

| Quan hệ | Điều kiện với ô `(r1, c1)` và `(r2, c2)` |
|---|---|
| cùng hàng | `r1 == r2` |
| cùng cột | `c1 == c2` |
| cùng đường chéo | `abs(r1 - r2) == abs(c1 - c2)` |

Ở đây `r` là **chỉ số hàng** (row, đếm từ 0 ở trên cùng), `c` là **chỉ số cột** (column, đếm từ 0 bên trái).

Để vẽ bàn cờ khi học, ta dùng một biểu diễn gọn: `queens[r] = c` nghĩa là "hậu ở hàng `r` nằm tại cột `c`". Lời giải A ở trên là `queens = [1, 3, 0, 2]`.

Hàm `print_board` dưới đây **chỉ để minh họa** — bài #52 không cần dựng bàn cờ, nhưng nhìn thấy bàn cờ giúp ta hiểu đề.

In [1]:
def print_board(queens):
    n = len(queens)
    for r in range(n):
        print(" ".join("Q" if queens[r] == c else "." for c in range(n)))

print("Lời giải A: queens = [1, 3, 0, 2]")
print_board([1, 3, 0, 2])
print()
print("Lời giải B: queens = [2, 0, 3, 1]")
print_board([2, 0, 3, 1])

Lời giải A: queens = [1, 3, 0, 2]
. Q . .
. . . Q
Q . . .
. . Q .

Lời giải B: queens = [2, 0, 3, 1]
. . Q .
Q . . .
. . . Q
. Q . .


## 3. Bài #52 khác bài #51 ở đâu?

| | #51 N-Queens | #52 N-Queens II (bài này) |
|---|---|---|
| Output | danh sách mọi bàn cờ, dạng `["..Q.", ...]` | **một số nguyên** |
| Phải lưu lời giải? | có, phải copy trạng thái khi gặp lời giải | **không**, chỉ cần `+1` |
| Phải dựng chuỗi? | có, tốn `O(n^2)` mỗi lời giải | **không** |
| Trọng tâm tối ưu | sinh output đúng định dạng | **làm phép kiểm tra và phép đếm thật rẻ** |

Vì vậy notebook này sẽ đi theo trục: *mỗi lần đứng ở một ô, mất bao nhiêu công để biết ô đó an toàn?* và *có phải duyệt cả bàn cờ không, hay chỉ một nửa?*

## 4. Nhận xét đầu tiên: mỗi hàng đúng một quân hậu

Có `n` hậu và `n` hàng. Hai hậu cùng hàng thì tấn công nhau, nên **mỗi hàng có tối đa 1 hậu**. Mà phải đặt đủ `n` hậu vào `n` hàng → **mỗi hàng có đúng 1 hậu**. Lập luận y hệt cho cột: **mỗi cột có đúng 1 hậu**.

Nhận xét này thu nhỏ không gian tìm kiếm cực mạnh:

| Cách mô tả lời giải | Số trường hợp phải xét |
|---|---|
| Chọn `n` ô bất kỳ trong `n^2` ô | `C(n^2, n)` |
| Mỗi hàng chọn 1 cột (`queens[r]` tùy ý) | `n^n` |
| Mỗi hàng 1 cột **và các cột khác nhau** | `n!` (giai thừa) |

Ta tính thử cho vài giá trị `n`:

In [2]:
import math

print(f"{'n':>2} | {'C(n*n, n)':>14} | {'n**n':>11} | {'n!':>7}")
for n in [4, 8, 9]:
    print(f"{n:>2} | {math.comb(n * n, n):>14,} | {n ** n:>11,} | {math.factorial(n):>7,}")

 n |      C(n*n, n) |        n**n |      n!
 4 |          1,820 |         256 |      24
 8 |  4,426,165,368 |  16,777,216 |  40,320
 9 | 260,887,834,350 | 387,420,489 | 362,880


Với `n = 9`: từ gần **261 tỷ** cách chọn ô xuống còn **362.880** hoán vị. Từ đây ta luôn mô tả một cách đặt hậu bằng `queens[r] = c`, và chỉ còn phải lo **cột** và **đường chéo**.

## 5. Kiến thức nền: đánh số đường chéo bằng `r - c` và `r + c`

Bàn cờ có 2 loại đường chéo:

- Chéo **"\\"** (đi xuống sang phải): khi `r` tăng 1 thì `c` cũng tăng 1, nên **`r - c` không đổi**.
- Chéo **"/"** (đi xuống sang trái): khi `r` tăng 1 thì `c` giảm 1, nên **`r + c` không đổi**.

Ví dụ: các ô `(0, 1)`, `(1, 2)`, `(2, 3)` có `r - c = -1` → cùng một chéo "\\". Các ô `(0, 3)`, `(1, 2)`, `(2, 1)`, `(3, 0)` có `r + c = 3` → cùng một chéo "/".

Nghĩa là mỗi đường chéo có một **mã số** riêng. Hai ô cùng chéo ⇔ cùng mã số. Ta in bảng mã số cho bàn `4 x 4`:

In [3]:
n = 4
print("Mã chéo '\\' = r - c        Mã chéo '/' = r + c")
for r in range(n):
    left = " ".join(f"{r - c:>2}" for c in range(n))
    right = " ".join(f"{r + c:>2}" for c in range(n))
    print(left, "      ", right)

Mã chéo '\' = r - c        Mã chéo '/' = r + c
 0 -1 -2 -3         0  1  2  3
 1  0 -1 -2         1  2  3  4
 2  1  0 -1         2  3  4  5
 3  2  1  0         3  4  5  6


Quan sát bảng:

- Bên trái, các số giống nhau nằm thẳng hàng theo hướng "\\". Mã từ `-(n-1)` tới `n-1` → có `2n - 1` đường chéo "\\".
- Bên phải, các số giống nhau nằm theo hướng "/". Mã từ `0` tới `2n-2` → cũng có `2n - 1` đường chéo "/".

Công thức `abs(r1 - r2) == abs(c1 - c2)` ở mục 2 và cặp công thức `r1 - c1 == r2 - c2` hoặc `r1 + c1 == r2 + c2` là **tương đương**. Ta sẽ dùng cách viết đầu ở cách 1–2, cách viết sau ở cách 3–4.

## 6. Cách 1 — Brute force: thử mọi hoán vị cột rồi lọc đường chéo

Theo mục 4, một lời giải là một **hoán vị** (permutation — cách sắp xếp lại thứ tự) của `[0, 1, ..., n-1]`: phần tử thứ `r` là cột của hậu hàng `r`. Hoán vị đã tự đảm bảo **khác hàng** và **khác cột**, ta chỉ cần lọc **đường chéo**.

Python có sẵn `itertools.permutations` để sinh mọi hoán vị:

In [4]:
from itertools import permutations

for p in permutations(range(3)):
    print(p)

(0, 1, 2)
(0, 2, 1)
(1, 0, 2)
(1, 2, 0)
(2, 0, 1)
(2, 1, 0)


3 phần tử → `3! = 6` hoán vị. Mỗi hoán vị là một `tuple` như `(1, 2, 0)`, đọc là "hàng 0 cột 1, hàng 1 cột 2, hàng 2 cột 0".

Hàm kiểm tra: xét mọi cặp hàng `i < j`, nếu `abs(p[i] - p[j]) == j - i` thì hai hậu cùng chéo (vì `j > i` nên `abs(i - j) = j - i`). Hàm demo trả về **cặp** `(số lời giải, số hoán vị đã xét)` để ta đo đạc.

In [5]:
def no_diagonal_attack(p):
    n = len(p)
    for i in range(n):
        for j in range(i + 1, n):
            if abs(p[i] - p[j]) == j - i:
                return False
    return True


def count_bruteforce(n):
    count = 0
    checked = 0
    for p in permutations(range(n)):
        checked += 1
        if no_diagonal_attack(p):
            count += 1
    return count, checked


for n in [1, 2, 3, 4, 5, 6]:
    print(f"n = {n}: (số lời giải, số hoán vị đã xét) = {count_bruteforce(n)}")

n = 1: (số lời giải, số hoán vị đã xét) = (1, 1)
n = 2: (số lời giải, số hoán vị đã xét) = (0, 2)
n = 3: (số lời giải, số hoán vị đã xét) = (0, 6)
n = 4: (số lời giải, số hoán vị đã xét) = (2, 24)
n = 5: (số lời giải, số hoán vị đã xét) = (10, 120)
n = 6: (số lời giải, số hoán vị đã xét) = (4, 720)


Kết quả `n = 4 → 2` và `n = 1 → 1` khớp đề. Nhưng nhìn cột thứ hai: số hoán vị phải xét luôn là `n!`.

### Cách 1 vướng ở đâu?

Hoán vị nào bắt đầu bằng `(0, 1, ...)` đều **chắc chắn sai** (hậu `(0,0)` và `(1,1)` cùng chéo), vậy mà brute force vẫn sinh ra **tất cả** chúng rồi mới kiểm tra. Với `n = 8`, số hoán vị như thế là `6! = 720` (6 hàng còn lại xếp tùy ý) — 720 lần kiểm tra vô ích chỉ cho một tiền tố hỏng.

Mấu chốt cải tiến: **đặt hậu từng hàng một, và kiểm tra ngay khi đặt**. Nếu hàng 1 đã xung đột với hàng 0 thì bỏ luôn cả nhánh, không sinh tiếp 6 hàng phía sau. Kỹ thuật "thử – nếu hỏng thì lùi lại" đó gọi là **backtracking**.

## 7. Kiến thức nền: recursion và backtracking từ căn bản

**Recursion** (đệ quy) là hàm tự gọi lại chính nó với bài toán nhỏ hơn. **Backtracking** (quay lui) là đệ quy dùng để thử từng lựa chọn theo 3 nhịp:

1. **Chọn** (choose): ghi nhận một lựa chọn vào trạng thái.
2. **Đi sâu** (explore): gọi đệ quy để giải phần còn lại.
3. **Bỏ chọn** (un-choose): xóa lựa chọn vừa ghi, trả trạng thái về như cũ để thử lựa chọn khác.

Ví dụ mini: sinh mọi cách xếp 3 cột `{0, 1, 2}` vào 3 hàng (chính là hoán vị) bằng backtracking. Mỗi mức thụt lề là một tầng đệ quy:

In [6]:
def demo_backtrack(path, n):
    indent = "  " * len(path)
    if len(path) == n:
        print(indent + f"-> đủ {n} hàng: {path}")
        return
    for c in range(n):
        if c in path:
            continue
        path.append(c)              # 1. chọn
        print(indent + f"chọn cột {c}, path = {path}")
        demo_backtrack(path, n)     # 2. đi sâu
        path.pop()                  # 3. bỏ chọn

demo_backtrack([], 3)

chọn cột 0, path = [0]
  chọn cột 1, path = [0, 1]
    chọn cột 2, path = [0, 1, 2]
      -> đủ 3 hàng: [0, 1, 2]
  chọn cột 2, path = [0, 2]
    chọn cột 1, path = [0, 2, 1]
      -> đủ 3 hàng: [0, 2, 1]
chọn cột 1, path = [1]
  chọn cột 0, path = [1, 0]
    chọn cột 2, path = [1, 0, 2]
      -> đủ 3 hàng: [1, 0, 2]
  chọn cột 2, path = [1, 2]
    chọn cột 0, path = [1, 2, 0]
      -> đủ 3 hàng: [1, 2, 0]
chọn cột 2, path = [2]
  chọn cột 0, path = [2, 0]
    chọn cột 1, path = [2, 0, 1]
      -> đủ 3 hàng: [2, 0, 1]
  chọn cột 1, path = [2, 1]
    chọn cột 0, path = [2, 1, 0]
      -> đủ 3 hàng: [2, 1, 0]


Hãy để ý: sau khi in `-> đủ 3 hàng: [0, 1, 2]`, hàm quay về tầng trên, `pop()` bỏ cột 2, rồi `pop()` bỏ cột 1, nên tầng 1 thử được cột 2 tiếp theo. Không có `pop()` thì `path` sẽ dài mãi và mọi thứ sai bét.

Toàn bộ quá trình tạo thành một **cây quyết định**: gốc là bàn trống, mỗi cạnh là một lựa chọn, mỗi lá ở độ sâu `n` là một lời giải. Mỗi lần gọi hàm đệ quy = thăm **một nút** của cây.

### Đếm bằng backtracking: cộng dồn kết quả các nhánh con

Bài #52 không cần in lời giải. Ta cho hàm đệ quy **trả về số lời giải trong cây con** của nó:

- Ở lá hợp lệ: trả về `1`.
- Ở nút trong: trả về **tổng** kết quả của các nhánh con.

Ví dụ mini: đếm số chuỗi nhị phân độ dài `k` **không có hai số 1 liền nhau** (vd `k = 3`: `000, 001, 010, 100, 101` → 5 chuỗi).

In [7]:
def count_no_11(k, prev=0):
    if k == 0:
        return 1                                   # lá: một chuỗi hợp lệ
    total = count_no_11(k - 1, 0)                  # nhánh đặt '0'
    if prev == 0:
        total += count_no_11(k - 1, 1)             # nhánh đặt '1' (chỉ khi trước đó không là 1)
    return total

for k in range(1, 6):
    print(f"k = {k}: {count_no_11(k)} chuỗi")

k = 1: 2 chuỗi
k = 2: 3 chuỗi
k = 3: 5 chuỗi
k = 4: 8 chuỗi
k = 5: 13 chuỗi


Nhánh `'1'` bị **cắt tỉa** (pruning) ngay khi ký tự trước là `1` — y hệt việc ta sẽ bỏ một cột khi nó bị tấn công. Và ta không cần lưu chuỗi nào cả, chỉ trả về con số. Đây đúng là tinh thần của bài #52.

## 8. Cách 2 — Backtracking, kiểm tra an toàn bằng cách quét các hàng trước

Trạng thái: mảng `queens` chứa cột của các hậu ở hàng `0, 1, ..., row-1` (đã đặt xong).

Hàm `backtrack(row)` trả về số lời giải khi đã đặt xong `row` hàng đầu:

- Nếu `row == n`: đã đặt đủ → trả về `1`.
- Ngược lại, thử từng cột `c` của hàng `row`. Ô `(row, c)` **an toàn** nếu với mọi hậu `(r2, queens[r2])` đã đặt: khác cột và không cùng chéo. Kiểm tra này phải **quét** toàn bộ `row` hậu trước → tốn `O(row)`.
- Nếu an toàn: chọn (`append`) → đi sâu → bỏ chọn (`pop`).

Hàm demo dùng biến `nodes` để đếm số lần gọi `backtrack` (= số nút thăm, tính cả gốc là bàn trống) và `checks` để đếm số phép so sánh hai hậu. `nonlocal` cho phép hàm con sửa biến của hàm cha.

In [8]:
def count_backtrack_scan(n):
    queens = []
    nodes = 0
    checks = 0

    def is_safe(row, c):
        nonlocal checks
        for r2, c2 in enumerate(queens):
            checks += 1
            if c2 == c or abs(r2 - row) == abs(c2 - c):
                return False
        return True

    def backtrack(row):
        nonlocal nodes
        nodes += 1
        if row == n:
            return 1
        total = 0
        for c in range(n):
            if is_safe(row, c):
                queens.append(c)          # chọn
                total += backtrack(row + 1)  # đi sâu, cộng số lời giải của nhánh con
                queens.pop()              # bỏ chọn
        return total

    return backtrack(0), nodes, checks


for n in [4, 8]:
    count, nodes, checks = count_backtrack_scan(n)
    print(f"n = {n}: lời giải = {count}, số nút = {nodes}, số phép so sánh = {checks}")
print("So sánh: brute force với n = 8 xét", math.factorial(8), "hoán vị")

n = 4: lời giải = 2, số nút = 17, số phép so sánh = 84
n = 8: lời giải = 92, số nút = 2057, số phép so sánh = 46752
So sánh: brute force với n = 8 xét 40320 hoán vị


Với `n = 8`, backtracking chỉ thăm **2.057 nút** thay vì sinh **40.320** hoán vị — cắt tỉa sớm phát huy tác dụng.

### Trace từng bước với `n = 4`

Hàm `trace_backtrack_scan` in **một dòng cho mỗi nút**: trạng thái `queens` hiện tại và kết quả kiểm tra **mọi cột** của hàng tiếp theo (`✓` = an toàn, `✗cột` / `✗chéo` = bị tấn công). Sau đó nó mới đi sâu vào các cột `✓` theo thứ tự trái → phải. Mức thụt lề = số hậu đã đặt.

In [9]:
def trace_backtrack_scan(n):
    queens = []
    found = 0

    def status(row, c):
        for r2, c2 in enumerate(queens):
            if c2 == c:
                return "✗cột"
            if abs(r2 - row) == abs(c2 - c):
                return "✗chéo"
        return "✓"

    def backtrack(row):
        nonlocal found
        indent = "  " * row
        if row == n:
            found += 1
            print(indent + f"queens={queens} -> LỜI GIẢI #{found}")
            return 1
        marks = [status(row, c) for c in range(n)]
        line = ", ".join(f"c{c}{m}" for c, m in enumerate(marks))
        dead = "  (ngõ cụt, quay lui)" if "✓" not in marks else ""
        print(indent + f"queens={queens} | hàng {row}: {line}{dead}")
        total = 0
        for c in range(n):
            if marks[c] == "✓":
                queens.append(c)
                total += backtrack(row + 1)
                queens.pop()
        return total

    return backtrack(0)

print("Tổng số lời giải:", trace_backtrack_scan(4))
print("Hàm đếm (lời giải, số nút, số phép so sánh):", count_backtrack_scan(4))

queens=[] | hàng 0: c0✓, c1✓, c2✓, c3✓
  queens=[0] | hàng 1: c0✗cột, c1✗chéo, c2✓, c3✓
    queens=[0, 2] | hàng 2: c0✗cột, c1✗chéo, c2✗chéo, c3✗chéo  (ngõ cụt, quay lui)
    queens=[0, 3] | hàng 2: c0✗cột, c1✓, c2✗chéo, c3✗cột
      queens=[0, 3, 1] | hàng 3: c0✗cột, c1✗chéo, c2✗chéo, c3✗chéo  (ngõ cụt, quay lui)
  queens=[1] | hàng 1: c0✗chéo, c1✗cột, c2✗chéo, c3✓
    queens=[1, 3] | hàng 2: c0✓, c1✗cột, c2✗chéo, c3✗chéo
      queens=[1, 3, 0] | hàng 3: c0✗cột, c1✗cột, c2✓, c3✗cột
        queens=[1, 3, 0, 2] -> LỜI GIẢI #1
  queens=[2] | hàng 1: c0✓, c1✗chéo, c2✗cột, c3✗chéo
    queens=[2, 0] | hàng 2: c0✗chéo, c1✗chéo, c2✗cột, c3✓
      queens=[2, 0, 3] | hàng 3: c0✗cột, c1✓, c2✗cột, c3✗cột
        queens=[2, 0, 3, 1] -> LỜI GIẢI #2
  queens=[3] | hàng 1: c0✓, c1✓, c2✗chéo, c3✗cột
    queens=[3, 0] | hàng 2: c0✗cột, c1✗chéo, c2✓, c3✗cột
      queens=[3, 0, 2] | hàng 3: c0✗chéo, c1✗chéo, c2✗chéo, c3✗cột  (ngõ cụt, quay lui)
    queens=[3, 1] | hàng 2: c0✗chéo, c1✗chéo, c2✗chéo, c3✗cộ

Đọc lại nhánh dẫn tới lời giải đầu tiên (đối chiếu với output trên):

| Bước | `queens` trước khi thử | Hàng thử | Kết quả kiểm tra các cột | Quyết định |
|---|---|---|---|---|
| 1 | `[]` | 0 | c0 ✓, c1 ✓, c2 ✓, c3 ✓ | thử lần lượt; nhánh c0 dẫn tới ngõ cụt (`[0,2]` và `[0,3,1]`) |
| 2 | `[1]` | 1 | c0 ✗chéo, c1 ✗cột, c2 ✗chéo, c3 ✓ | chỉ còn c3 |
| 3 | `[1, 3]` | 2 | c0 ✓, c1 ✗cột, c2 ✗chéo, c3 ✗chéo (*) | chỉ còn c0 |
| 4 | `[1, 3, 0]` | 3 | c0 ✗cột, c1 ✗cột, c2 ✓, c3 ✗cột | chỉ còn c2 |
| 5 | `[1, 3, 0, 2]` | — | đủ 4 hậu | **lời giải #1**, trả về 1 |

(*) Cột 3 vừa trùng cột với hậu hàng 1, vừa cùng chéo với hậu `(0, 1)`; hàm `status` quét từ hàng 0 nên báo lý do **đầu tiên** nó gặp là `✗chéo`.

Nhận xét: các nút ngõ cụt (`[0, 2]`, `[0, 3, 1]`, `[3, 0, 2]`, `[3, 1]`) bị bỏ ngay — đó là phần việc brute force phải làm thừa. Đếm số dòng có `queens=` ta được **17 nút** (tính cả 2 dòng LỜI GIẢI), khớp với dòng cuối của output do `count_backtrack_scan(4)` in ra.

## 9. Cách 3 — Backtracking với 3 `set`: kiểm tra an toàn trong `O(1)`

### Cách 2 vướng ở đâu?

Số nút đã ít, nhưng **mỗi lần kiểm tra một ô** lại quét lại toàn bộ hậu đã đặt. Với `n = 8` đó là **46.752** phép so sánh cho 2.057 nút (trung bình hơn 22 phép mỗi nút). Ta đang hỏi đi hỏi lại cùng một câu "cột 3 đã có ai chưa?".

### Mấu chốt cải tiến

Ghi nhớ sẵn các cột và đường chéo **đã bị chiếm**:

- `cols`: các cột đã có hậu.
- `diag1`: các mã `r - c` (chéo "\\") đã có hậu.
- `diag2`: các mã `r + c` (chéo "/") đã có hậu.

Ô `(r, c)` an toàn ⇔ `c not in cols and (r - c) not in diag1 and (r + c) not in diag2`. Ba phép tra cứu, không phụ thuộc số hậu đã đặt.

### Kiến thức nền: `set` trong Python

`set` (tập hợp) lưu các giá trị **không trùng lặp**; `add` thêm, `remove` xóa, `x in s` kiểm tra có hay không — trung bình `O(1)` nhờ cơ chế băm (hash).

Trong code cách 3 (cell thứ hai bên dưới), phần "chọn" là **thêm vào 3 set**, phần "bỏ chọn" là **xóa khỏi 3 set**. Ta **không cần mảng `queens`** nữa — vì chỉ đếm, không phải in bàn cờ.

In [10]:
diag1 = set()
diag1.add(0 - 1)        # đặt hậu tại (0, 1): mã chéo '\' là -1
print("diag1 =", diag1)
print("(1, 2) cùng chéo '\\' với (0, 1)?", (1 - 2) in diag1)   # -1 -> True
print("(1, 0) cùng chéo '\\' với (0, 1)?", (1 - 0) in diag1)   #  1 -> False
diag1.remove(-1)        # bỏ chọn
print("sau remove:", diag1)

diag1 = {-1}
(1, 2) cùng chéo '\' với (0, 1)? True
(1, 0) cùng chéo '\' với (0, 1)? False
sau remove: set()


In [11]:
def count_backtrack_sets(n):
    cols, diag1, diag2 = set(), set(), set()
    nodes = 0

    def backtrack(r):
        nonlocal nodes
        nodes += 1
        if r == n:
            return 1
        total = 0
        for c in range(n):
            if c in cols or (r - c) in diag1 or (r + c) in diag2:
                continue
            cols.add(c); diag1.add(r - c); diag2.add(r + c)            # chọn
            total += backtrack(r + 1)                                  # đi sâu
            cols.remove(c); diag1.remove(r - c); diag2.remove(r + c)   # bỏ chọn
        return total

    return backtrack(0), nodes


for n in [1, 4, 8, 9]:
    print(f"n = {n}: (lời giải, số nút) = {count_backtrack_sets(n)}")

n = 1: (lời giải, số nút) = (1, 2)
n = 4: (lời giải, số nút) = (2, 17)
n = 8: (lời giải, số nút) = (92, 2057)
n = 9: (lời giải, số nút) = (352, 8394)


**Số nút giống hệt cách 2** (`n = 8` vẫn 2.057): ta không cắt tỉa thêm gì, chỉ làm **mỗi nút rẻ hơn**. Mỗi nút giờ chỉ tốn `n` lần thử cột × 3 tra cứu `O(1)`.

Ta đo thời gian thực để thấy khác biệt (con số cụ thể tùy máy, hãy nhìn **tỉ lệ**):

In [12]:
import time

def time_it(func, n, repeat=3):
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        func(n)
        best = min(best, time.perf_counter() - start)
    return best

n = 9
t_scan = time_it(count_backtrack_scan, n)
t_sets = time_it(count_backtrack_sets, n)
print(f"n = {n}: quét = {t_scan * 1000:.1f} ms, 3 set = {t_sets * 1000:.1f} ms")
print(f"3 set nhanh hơn khoảng {t_scan / t_sets:.1f} lần")

n = 9: quét = 25.8 ms, 3 set = 7.8 ms
3 set nhanh hơn khoảng 3.3 lần


## 10. Kiến thức nền: phép toán bit từ số 0

Cách 3 đã tốt, nhưng mỗi nút vẫn **thử đủ `n` cột** rồi mới loại, và `set` có chi phí băm. Ý tưởng tiếp theo: vì `n <= 9`, mỗi tập "cột bị chiếm" có thể nén vào **một số nguyên**, trong đó **bit thứ `c` bằng 1 nghĩa là cột `c` bị chiếm**. Cách biểu diễn tập hợp bằng các bit như vậy gọi là **bitmask** (mặt nạ bit).

Trước hết, ôn số nhị phân. Số `13` viết nhị phân là `1101`:

```text
vị trí bit:   3 2 1 0
giá trị bit:  1 1 0 1   ->  8 + 4 + 0 + 1 = 13
```

Bit số `0` là bit **phải nhất** (bit thấp nhất). Python: `bin(x)` in dạng nhị phân, `0b1101` là cách viết số nhị phân trực tiếp, `format(x, "04b")` in đủ 4 chữ số.

| Toán tử | Tên | Ví dụ (4 bit) | Ý nghĩa với tập hợp |
|---|---|---|---|
| `1 << k` | dịch trái | `1 << 2 = 0100` | tập chỉ có phần tử `k` |
| `a \| b` | OR | `0101 \| 0011 = 0111` | hợp hai tập |
| `a & b` | AND | `0101 & 0011 = 0001` | giao hai tập |
| `~a` | NOT | đảo mọi bit | phần bù (cẩn thận, xem dưới) |
| `a ^ b` | XOR | `0101 ^ 0001 = 0100` | bật/tắt bit |
| `a >> 1` | dịch phải | `0110 >> 1 = 0011` | mọi phần tử giảm 1 |

In [13]:
a, b = 0b0101, 0b0011
print("a      =", format(a, "04b"), "=", a)
print("b      =", format(b, "04b"), "=", b)
print("1 << 2 =", format(1 << 2, "04b"))
print("a | b  =", format(a | b, "04b"))
print("a & b  =", format(a & b, "04b"))
print("a ^ 1  =", format(a ^ 1, "04b"))
print("a << 1 =", format(a << 1, "04b"))
print("a >> 1 =", format(a >> 1, "04b"))

a      = 0101 = 5
b      = 0011 = 3
1 << 2 = 0100
a | b  = 0111
a & b  = 0001
a ^ 1  = 0100
a << 1 = 1010
a >> 1 = 0010


### Dùng số nguyên như một tập hợp cột

Với bàn `n = 4`, giả sử cột 0 và cột 2 đã bị chiếm: `cols = 0b0101`.

- Kiểm tra cột `c` có bị chiếm: `(cols >> c) & 1` hoặc `(cols & (1 << c)) != 0`.
- Đánh dấu cột `c`: `cols | (1 << c)`.

**Cẩn thận khi đọc**: chuỗi nhị phân in **bit cao bên trái**, nên cột 0 nằm **bên phải nhất** — ngược với bàn cờ. Để đỡ rối, ta viết hàm `board_view(mask, n)` in các cột theo thứ tự bàn cờ `0, 1, ..., n-1` từ trái sang phải (`x` = bị chiếm, `.` = trống).

### `~` và mặt nạ đầy `full`

Trong Python số nguyên không giới hạn bit, nên `~x` = `-x - 1` là một **số âm** (về mặt bit: vô hạn số 1 ở bên trái). Muốn "phần bù trong phạm vi `n` cột", ta AND với `full = (1 << n) - 1` = `n` bit 1:

```text
n = 4: full = (1 << 4) - 1 = 10000 - 1 = 1111
```

In [14]:
def board_view(mask, n):
    return "".join("x" if (mask >> c) & 1 else "." for c in range(n))

n = 4
cols = 0b0101
print("nhị phân:", format(cols, "04b"), "| nhìn theo bàn cờ (cột 0 -> 3):", board_view(cols, n))
for c in range(n):
    print(f"cột {c} bị chiếm? {(cols >> c) & 1}")
cols = cols | (1 << 3)
print("sau khi chiếm thêm cột 3:", format(cols, "04b"), "|", board_view(cols, n))
print()

full = (1 << n) - 1
cols = 0b0101
print("full         =", format(full, "04b"))
print("~cols        =", ~cols, "(số âm, không dùng trực tiếp được)")
print("full & ~cols =", format(full & ~cols, "04b"), "-> các cột còn trống:", board_view(full & ~cols, n))

nhị phân: 0101 | nhìn theo bàn cờ (cột 0 -> 3): x.x.
cột 0 bị chiếm? 1
cột 1 bị chiếm? 0
cột 2 bị chiếm? 1
cột 3 bị chiếm? 0
sau khi chiếm thêm cột 3: 1101 | x.xx

full         = 1111
~cols        = -6 (số âm, không dùng trực tiếp được)
full & ~cols = 1010 -> các cột còn trống: .x.x


### Mẹo `x & -x`: lấy bit 1 thấp nhất

Trong biểu diễn bù hai (two's complement — cách máy tính lưu số âm), `-x = ~x + 1`. Ví dụ `x = 12 = 1100`:

```text
 x      = ...0001100
~x      = ...1110011
~x + 1  = ...1110100   (= -x)
x & -x  = ...0000100   -> chỉ còn bit 1 thấp nhất của x
```

Vì sao? `~x` đảo mọi bit; `+1` làm các bit 1 ở cuối của `~x` (tức các bit 0 ở cuối của `x`) chuyển thành 0 và "nhớ" sang đúng vị trí bit 1 thấp nhất của `x`. Chỉ tại vị trí đó `x` và `-x` cùng là 1.

Cặp mẹo đi kèm:

- `bit = x & -x` → lấy bit thấp nhất.
- `x ^= bit` (hoặc `x &= x - 1`) → xóa bit đó khỏi `x`.

Lặp hai bước này giúp ta **chỉ đi qua các bit 1**, không phải dò từng vị trí.

In [15]:
x = 0b1100
print("x       =", format(x, "04b"))
print("x & -x  =", format(x & -x, "04b"))
print("x & x-1 =", format(x & (x - 1), "04b"), "(xóa bit thấp nhất)")
print()

available = 0b1011
print("Duyệt các bit 1 của", format(available, "04b"))
while available:
    bit = available & -available
    available ^= bit
    print(f"  bit = {format(bit, '04b')} -> cột {bit.bit_length() - 1}, còn lại {format(available, '04b')}")

x       = 1100
x & -x  = 0100
x & x-1 = 1000 (xóa bit thấp nhất)

Duyệt các bit 1 của 1011
  bit = 0001 -> cột 0, còn lại 1010
  bit = 0010 -> cột 1, còn lại 1000
  bit = 1000 -> cột 3, còn lại 0000


`bit.bit_length() - 1` cho ra chỉ số cột của `bit` (vd `0b0100` có `bit_length = 3` → cột 2). Khi chỉ cần **đếm**, ta thậm chí không cần biết chỉ số cột — chỉ cần chính con số `bit`.

## 11. Cách 4 — Bitmask: đường chéo "trượt" khi xuống hàng

Ta giữ 3 bitmask **cho hàng đang xét**:

- `cols`: bit `c` = 1 nếu cột `c` đã có hậu.
- `d1`: bit `c` = 1 nếu ô `(hàng hiện tại, c)` bị một hậu phía trên tấn công theo chéo **"\\"**.
- `d2`: bit `c` = 1 nếu ô `(hàng hiện tại, c)` bị tấn công theo chéo **"/"**.

Điểm hay: không cần mã `r - c` nữa. Một hậu ở `(r, c)` tấn công theo chéo "\\" ô `(r+1, c+1)` ở hàng dưới, rồi `(r+2, c+2)`... Tức là **mỗi lần xuống một hàng, vết tấn công "\\" dịch sang cột lớn hơn 1** → `d1 << 1`. Vết tấn công "/" dịch sang cột nhỏ hơn 1 → `d2 >> 1`. Bit trượt ra ngoài bàn cờ thì cắt đi bằng `& full`.

Ở mỗi hàng:

```python
available = full & ~(cols | d1 | d2)     # các cột an toàn, tính 1 lần cho cả hàng
while available:
    bit = available & -available          # chọn cột an toàn thấp nhất
    available ^= bit
    total += go(cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)
```

Phần "bỏ chọn" **biến mất**: ta truyền bitmask mới làm tham số, các biến của tầng hiện tại không hề bị sửa. Và dừng khi `cols == full` (đủ `n` cột có hậu).

Xem một bước cụ thể với `n = 4`, hậu hàng 0 đặt ở cột 1:

In [16]:
n = 4
full = (1 << n) - 1
cols, d1, d2 = 0, 0, 0
bit = 1 << 1                                    # hàng 0, cột 1

cols, d1, d2 = cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1
print("Sang hàng 1 (cột 0 -> 3 từ trái sang):")
print("  cols      =", board_view(cols, n), " (cột 1 bị chiếm)")
print("  d1 (\\)    =", board_view(d1, n), " (chéo '\\' từ (0,1) chạm (1,2))")
print("  d2 (/)    =", board_view(d2, n), " (chéo '/' từ (0,1) chạm (1,0))")
available = full & ~(cols | d1 | d2)
print("  available =", board_view(available, n).replace("x", "o"), " ('o' = an toàn) -> chỉ còn cột 3")

Sang hàng 1 (cột 0 -> 3 từ trái sang):
  cols      = .x..  (cột 1 bị chiếm)
  d1 (\)    = ..x.  (chéo '\' từ (0,1) chạm (1,2))
  d2 (/)    = x...  (chéo '/' từ (0,1) chạm (1,0))
  available = ...o  ('o' = an toàn) -> chỉ còn cột 3


In [17]:
# Khớp với trace cách 2 ở bước 2: queens = [1] thì hàng 1 chỉ còn cột 3.
# Giờ là code đầy đủ của cách 4:
def count_bitmask(n):
    full = (1 << n) - 1
    nodes = 0

    def go(cols, d1, d2):
        nonlocal nodes
        nodes += 1
        if cols == full:
            return 1
        total = 0
        available = full & ~(cols | d1 | d2)
        while available:
            bit = available & -available
            available ^= bit
            total += go(cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)
        return total

    return go(0, 0, 0), nodes


for n in [1, 4, 8, 9]:
    print(f"n = {n}: (lời giải, số nút) = {count_bitmask(n)}")

n = 1: (lời giải, số nút) = (1, 2)
n = 4: (lời giải, số nút) = (2, 17)
n = 8: (lời giải, số nút) = (92, 2057)
n = 9: (lời giải, số nút) = (352, 8394)


### Trace cách 4 với `n = 4`

Mỗi dòng là một nút: 3 mặt nạ của hàng đang xét và `available` (cột an toàn, `o`). So sánh với trace cách 2: **cùng một cây**, nhưng giờ cả hàng được kiểm tra bằng **một biểu thức**, và vòng lặp chỉ đi qua cột `o`.

In [18]:
def trace_bitmask(n):
    full = (1 << n) - 1
    found = 0

    def go(row, cols, d1, d2):
        nonlocal found
        indent = "  " * row
        if cols == full:
            found += 1
            print(indent + f"cols={board_view(cols, n)} -> LỜI GIẢI #{found}")
            return 1
        available = full & ~(cols | d1 | d2)
        avail_txt = board_view(available, n).replace("x", "o")
        cut = "  (ngõ cụt)" if available == 0 else ""
        print(indent + f"hàng {row}: cols={board_view(cols, n)} d1={board_view(d1, n)} "
              f"d2={board_view(d2, n)} -> available={avail_txt}{cut}")
        total = 0
        while available:
            bit = available & -available
            available ^= bit
            total += go(row + 1, cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)
        return total

    return go(0, 0, 0, 0)

print("Tổng:", trace_bitmask(4))

hàng 0: cols=.... d1=.... d2=.... -> available=oooo
  hàng 1: cols=x... d1=.x.. d2=.... -> available=..oo
    hàng 2: cols=x.x. d1=..xx d2=.x.. -> available=....  (ngõ cụt)
    hàng 2: cols=x..x d1=..x. d2=..x. -> available=.o..
      hàng 3: cols=xx.x d1=..xx d2=xx.. -> available=....  (ngõ cụt)
  hàng 1: cols=.x.. d1=..x. d2=x... -> available=...o
    hàng 2: cols=.x.x d1=...x d2=..x. -> available=o...
      hàng 3: cols=xx.x d1=.x.. d2=.x.. -> available=..o.
        cols=xxxx -> LỜI GIẢI #1
  hàng 1: cols=..x. d1=...x d2=.x.. -> available=o...
    hàng 2: cols=x.x. d1=.x.. d2=x... -> available=...o
      hàng 3: cols=x.xx d1=..x. d2=..x. -> available=.o..
        cols=xxxx -> LỜI GIẢI #2
  hàng 1: cols=...x d1=.... d2=..x. -> available=oo..
    hàng 2: cols=x..x d1=.x.. d2=.x.. -> available=..o.
      hàng 3: cols=x.xx d1=..xx d2=xx.. -> available=....  (ngõ cụt)
    hàng 2: cols=.x.x d1=..x. d2=xx.. -> available=....  (ngõ cụt)
Tổng: 2


Đọc nhánh lời giải #1 từ output:

| Hàng | `cols` | `d1` (\\) | `d2` (/) | `available` | Chọn |
|---|---|---|---|---|---|
| 0 | `....` | `....` | `....` | `oooo` | thử cột 0 (ngõ cụt), rồi **cột 1** |
| 1 | `.x..` | `..x.` | `x...` | `...o` | cột 3 |
| 2 | `.x.x` | `...x` | `..x.` | `o...` | cột 0 |
| 3 | `xx.x` | `.x..` | `.x..` | `..o.` | cột 2 |
| 4 | `xxxx` | | | | **lời giải #1** |

Ở hàng 3: `d1 = .x..` vì hậu `(2,0)` tấn công chéo "\\" ô `(3,1)`; `d2 = .x..` vì hậu `(1,3)` tấn công chéo "/" ô `(3,1)` (vết "/" của `(1,3)` đi `(2,2)` rồi `(3,1)`), còn vết "\\" của `(0,1)` đi `(1,2)`, `(2,3)` rồi trượt ra ngoài và bị `& full` cắt.

## 12. Cách 5 — Đối xứng gương: chỉ duyệt nửa cây

### Cách 4 còn thừa ở đâu?

Nhớ lại ví dụ đề: lời giải B `[2, 0, 3, 1]` là ảnh **lật gương trái–phải** của lời giải A `[1, 3, 0, 2]` (cột `c` ↦ cột `n-1-c`). Điều này luôn đúng: lật gương không làm đổi quan hệ cùng cột / cùng chéo, nên **lật một lời giải luôn được một lời giải**.

### Mấu chốt

- Lời giải có hậu hàng 0 ở cột `c` ↔ lời giải lật có hậu hàng 0 ở cột `n-1-c`. Đây là một **cặp ghép đôi 1–1**.
- Vậy số lời giải bắt đầu ở cột `c` **bằng** số lời giải bắt đầu ở cột `n-1-c`.
- ⇒ Chỉ cần duyệt hàng 0 ở các cột `0 .. n//2 - 1` (nửa trái), rồi **nhân 2**.
- Nếu `n` **lẻ**, còn cột giữa `n//2` tự đối xứng với chính nó → duyệt riêng và **không nhân 2**.

Kiểm chứng bằng số liệu: đếm số lời giải theo cột của hậu hàng 0.

In [19]:
def count_by_first_column(n):
    full = (1 << n) - 1

    def go(cols, d1, d2):
        if cols == full:
            return 1
        total = 0
        available = full & ~(cols | d1 | d2)
        while available:
            bit = available & -available
            available ^= bit
            total += go(cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)
        return total

    result = []
    for c in range(n):
        bit = 1 << c
        result.append(go(bit, (bit << 1) & full, bit >> 1))
    return result

for n in [4, 5, 6, 8]:
    by_col = count_by_first_column(n)
    print(f"n = {n}: theo cột hàng 0 = {by_col}, tổng = {sum(by_col)}")

n = 4: theo cột hàng 0 = [0, 1, 1, 0], tổng = 2
n = 5: theo cột hàng 0 = [2, 2, 2, 2, 2], tổng = 10
n = 6: theo cột hàng 0 = [0, 1, 1, 1, 1, 0], tổng = 4
n = 8: theo cột hàng 0 = [4, 8, 16, 18, 18, 16, 8, 4], tổng = 92


Danh sách luôn **đối xứng** (đọc xuôi = đọc ngược), ví dụ `n = 8`: `[4, 8, 16, 18, 18, 16, 8, 4]`. Với `n = 5` (lẻ): `[2, 2, 2, 2, 2]` — cột giữa (cột 2) đóng góp 2 lời giải, chỉ được cộng **một lần**.

Code cách 5 (xây trên cách 4). Vì hàng 0 được đặt tay, ta tính sẵn mặt nạ của hàng 1 rồi gọi `go`; `nodes` bắt đầu từ 1 để tính cả nút gốc (bàn trống) cho công bằng khi so sánh:

In [20]:
def count_bitmask_symmetry(n):
    full = (1 << n) - 1
    nodes = 1                     # nút gốc: bàn trống

    def go(cols, d1, d2):
        nonlocal nodes
        nodes += 1
        if cols == full:
            return 1
        total = 0
        available = full & ~(cols | d1 | d2)
        while available:
            bit = available & -available
            available ^= bit
            total += go(cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)
        return total

    total = 0
    for c in range(n // 2):                      # nửa trái: nhân 2
        bit = 1 << c
        total += 2 * go(bit, (bit << 1) & full, bit >> 1)
    if n % 2 == 1:                               # cột giữa khi n lẻ: không nhân 2
        bit = 1 << (n // 2)
        total += go(bit, (bit << 1) & full, bit >> 1)
    return total, nodes


for n in [1, 2, 3, 4, 5, 8, 9]:
    print(f"n = {n}: (lời giải, số nút) = {count_bitmask_symmetry(n)}")

n = 1: (lời giải, số nút) = (1, 2)
n = 2: (lời giải, số nút) = (0, 2)
n = 3: (lời giải, số nút) = (0, 4)
n = 4: (lời giải, số nút) = (2, 9)
n = 5: (lời giải, số nút) = (10, 32)
n = 8: (lời giải, số nút) = (92, 1029)
n = 9: (lời giải, số nút) = (352, 4704)


`n = 8`: còn **1.029 nút** thay vì 2.057 — gần đúng một nửa. Với `n` lẻ thì tiết kiệm ít hơn một nửa vì cột giữa vẫn phải duyệt đủ (`n = 9`: 4.704 so với 8.394).

Trường hợp biên: `n = 1` → vòng `for` không chạy (`1 // 2 = 0`), chỉ có cột giữa → 1. `n = 2`, `n = 3` → 0 lời giải, không phải lỗi.

## 13. Bảng kết quả `n = 1..9` và số nút duyệt của từng cách

- **Brute force**: số hoán vị phải kiểm tra (`n!`).
- **Backtracking** (cách 2, 3, 4): số lần gọi hàm đệ quy — ba cách thăm **cùng một cây** nên số nút bằng nhau; khác nhau ở **giá mỗi nút**.
- **Đối xứng** (cách 5): số nút khi chỉ duyệt nửa cây.

Phần sau của cell còn đo thời gian thực cho `n = 9` (lấy giá trị nhỏ nhất trong vài lần chạy; con số tùy máy, hãy nhìn thứ tự và tỉ lệ).

In [21]:
print(f"{'n':>2} | {'kết quả':>7} | {'brute (n!)':>10} | {'scan':>5} | {'sets':>5} | {'bitmask':>7} | {'đối xứng':>8}")
print("-" * 62)
for n in range(1, 10):
    r1, perms = count_bruteforce(n)
    r2, nodes2, _ = count_backtrack_scan(n)
    r3, nodes3 = count_backtrack_sets(n)
    r4, nodes4 = count_bitmask(n)
    r5, nodes5 = count_bitmask_symmetry(n)
    assert r1 == r2 == r3 == r4 == r5
    assert nodes2 == nodes3 == nodes4
    print(f"{n:>2} | {r4:>7} | {perms:>10,} | {nodes2:>5,} | {nodes3:>5,} | {nodes4:>7,} | {nodes5:>8,}")

print()
print("Thời gian thực với n = 9:")
n = 9
results = [
    ("1. brute force hoán vị", time_it(count_bruteforce, n, repeat=1)),
    ("2. backtracking + quét", time_it(count_backtrack_scan, n)),
    ("3. backtracking + 3 set", time_it(count_backtrack_sets, n)),
    ("4. bitmask", time_it(count_bitmask, n)),
    ("5. bitmask + đối xứng", time_it(count_bitmask_symmetry, n)),
]
base = results[0][1]
for name, t in results:
    print(f"{name:<26} {t * 1000:8.2f} ms   (nhanh gấp ~{base / t:5.1f} lần brute force)")

 n | kết quả | brute (n!) |  scan |  sets | bitmask | đối xứng
--------------------------------------------------------------
 1 |       1 |          1 |     2 |     2 |       2 |        2
 2 |       0 |          2 |     3 |     3 |       3 |        2
 3 |       0 |          6 |     6 |     6 |       6 |        4
 4 |       2 |         24 |    17 |    17 |      17 |        9
 5 |      10 |        120 |    54 |    54 |      54 |       32
 6 |       4 |        720 |   153 |   153 |     153 |       77
 7 |      40 |      5,040 |   552 |   552 |     552 |      316


 8 |      92 |     40,320 | 2,057 | 2,057 |   2,057 |    1,029


 9 |     352 |    362,880 | 8,394 | 8,394 |   8,394 |    4,704

Thời gian thực với n = 9:


1. brute force hoán vị       227.83 ms   (nhanh gấp ~  1.0 lần brute force)
2. backtracking + quét        25.20 ms   (nhanh gấp ~  9.0 lần brute force)
3. backtracking + 3 set        7.27 ms   (nhanh gấp ~ 31.3 lần brute force)
4. bitmask                     1.89 ms   (nhanh gấp ~120.3 lần brute force)
5. bitmask + đối xứng          1.06 ms   (nhanh gấp ~214.3 lần brute force)


## 14. Code nộp LeetCode

Bản chính: **bitmask + đối xứng** (cách 4 + 5). Bỏ hết biến đếm nút, chỉ giữ phần cần thiết. Hàm `go` lồng trong method để dùng chung `full`.

Nếu bạn thấy bitmask còn lạ tay, bản phụ `SolutionSets` (cách 3, ở cell thứ hai) dễ nhớ và cũng nộp được thoải mái với `n <= 9` — khi nộp chỉ cần đổi tên class thành `Solution`.

In [22]:
class Solution:
    def totalNQueens(self, n: int) -> int:
        full = (1 << n) - 1

        def go(cols: int, d1: int, d2: int) -> int:
            if cols == full:
                return 1
            total = 0
            available = full & ~(cols | d1 | d2)
            while available:
                bit = available & -available
                available ^= bit
                total += go(cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)
            return total

        total = 0
        for c in range(n // 2):
            bit = 1 << c
            total += 2 * go(bit, (bit << 1) & full, bit >> 1)
        if n % 2 == 1:
            bit = 1 << (n // 2)
            total += go(bit, (bit << 1) & full, bit >> 1)
        return total


sol = Solution()
assert sol.totalNQueens(4) == 2   # Example 1
assert sol.totalNQueens(1) == 1   # Example 2
print("Ví dụ đề: PASS")

Ví dụ đề: PASS


In [23]:
class SolutionSets:
    def totalNQueens(self, n: int) -> int:
        cols, diag1, diag2 = set(), set(), set()

        def backtrack(r: int) -> int:
            if r == n:
                return 1
            total = 0
            for c in range(n):
                if c in cols or (r - c) in diag1 or (r + c) in diag2:
                    continue
                cols.add(c); diag1.add(r - c); diag2.add(r + c)
                total += backtrack(r + 1)
                cols.remove(c); diag1.remove(r - c); diag2.remove(r + c)
            return total

        return backtrack(0)


assert SolutionSets().totalNQueens(4) == 2
assert SolutionSets().totalNQueens(1) == 1
print("SolutionSets: PASS")

SolutionSets: PASS


### Bộ test đầy đủ

1. Toàn bộ miền input `n = 1..9` so với giá trị đã biết `[1, 0, 0, 2, 10, 4, 40, 92, 352]` (dãy kinh điển của bài toán N quân hậu).
2. Edge case: `n = 1` (một ô), `n = 2, 3` (không có lời giải), `n` chẵn/lẻ (nhánh cột giữa).
3. Đối chiếu chéo **5 cách** theo thứ tự ngẫu nhiên có `random.seed` cố định, và gọi lặp lại trên **cùng một object** `Solution` để chắc không có trạng thái sót lại giữa các lần gọi.

In [24]:
import random

EXPECTED = [1, 0, 0, 2, 10, 4, 40, 92, 352]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)

sol = Solution()
for n in range(1, 10):
    check(f"Solution n={n}", sol.totalNQueens(n), EXPECTED[n - 1])
    check(f"SolutionSets n={n}", SolutionSets().totalNQueens(n), EXPECTED[n - 1])
print("Miền input 1..9: PASS")

check("n=1", sol.totalNQueens(1), 1)
check("n=2", sol.totalNQueens(2), 0)
check("n=3", sol.totalNQueens(3), 0)
print("Edge case: PASS")

random.seed(52)
order = [random.randint(1, 8) for _ in range(12)]
print("Thứ tự ngẫu nhiên:", order)
for n in order:
    answers = {
        sol.totalNQueens(n),
        count_bruteforce(n)[0],
        count_backtrack_scan(n)[0],
        count_backtrack_sets(n)[0],
        count_bitmask(n)[0],
        count_bitmask_symmetry(n)[0],
    }
    assert len(answers) == 1, (n, answers)
print("Đối chiếu chéo 5 cách + Solution: PASS")

Miền input 1..9: PASS
Edge case: PASS
Thứ tự ngẫu nhiên: [5, 1, 8, 6, 7, 1, 3, 3, 7, 3, 6, 8]
Đối chiếu chéo 5 cách + Solution: PASS


## 15. Vì sao đúng?

**Invariant** (bất biến — điều luôn đúng mỗi khi vào hàm đệ quy) của cách 2–4:

> Khi vào `backtrack(r)` / `go(...)`, các hàng `0 .. r-1` đã có đúng một hậu mỗi hàng, **không hai hậu nào tấn công nhau**, và trạng thái (`queens` / 3 set / 3 bitmask) mô tả **chính xác** các cột và đường chéo mà những hậu đó chiếm.

- **Không bỏ sót**: ở mỗi hàng ta thử **mọi** cột, chỉ loại những cột bị tấn công. Một cột bị tấn công thì không lời giải nào đi qua nó được, nên loại không làm mất lời giải.
- **Không đếm trùng**: hai nhánh khác nhau khác cột ở một hàng nào đó → hai bàn cờ khác nhau. Mỗi lời giải ứng với **đúng một** đường đi từ gốc tới lá.
- **Bỏ chọn đúng**: ở cách 2–3, sau khi nhánh con trả về ta xóa đúng những gì đã thêm, nên invariant đúng trở lại cho cột kế tiếp. Ở cách 4, bitmask được truyền theo giá trị nên tự động đúng.
- **Bitmask dịch đúng**: vết chéo "\\" của hậu `(r, c)` ở hàng `r + k` nằm tại cột `c + k` → dịch trái `k` lần; vết "/" nằm tại `c - k` → dịch phải `k` lần. Mỗi tầng dịch thêm 1 nên ở hàng nào `d1`, `d2` cũng chứa đúng vết của mọi hậu phía trên.
- **Đối xứng đúng**: lật gương là song ánh (bijection — ghép đôi 1–1) giữa lời giải bắt đầu ở cột `c` và lời giải bắt đầu ở cột `n-1-c`. Với `c < n//2` hai cột này khác nhau nên hai nhóm rời nhau, cộng `2 ×`. Cột giữa (`n` lẻ) ghép đôi với chính nó nên đã được đếm đủ trong nhánh của nó.

## 16. Độ phức tạp

Gọi `n` là kích thước bàn cờ.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Brute force hoán vị | `O(n! · n^2)` | `O(n)` | luôn sinh đủ `n!` hoán vị, mỗi hoán vị kiểm tra tối đa `n(n-1)/2` cặp |
| 2. Backtracking + quét | `O(n!)` nút × `O(n^2)` mỗi nút | `O(n)` | mỗi nút thử `n` cột, mỗi cột quét `O(n)` hậu |
| 3. Backtracking + 3 set | `O(n!)` nút × `O(n)` mỗi nút | `O(n)` | 3 set tối đa `n` phần tử + stack đệ quy sâu `n` |
| 4. Bitmask | `O(n!)` nút × `O(1)` mỗi nhánh con | `O(n)` | một biểu thức cho cả hàng, chỉ lặp qua cột an toàn; phép bit trên số ≤ `n` bit |
| 5. Bitmask + đối xứng | như cách 4, hằng số ≈ ½ | `O(n)` | |

Vì sao cận trên là `O(n!)` nút? Hàng `r` có tối đa `n - r` cột chưa bị chiếm, nên số nút ở độ sâu `k` không quá `n · (n-1) · ... · (n-k+1)`; cộng lại vẫn `≤ e · n!`. Thực tế nhỏ hơn rất nhiều nhờ cắt tỉa đường chéo (bảng mục 13: `n = 9` chỉ 8.394 nút so với `9! = 362.880`). Với `n <= 9`, mọi cách 2–5 đều chạy tức thì.

Bộ nhớ: không có cách nào lưu lời giải — đây là lợi thế của "chỉ đếm" so với bài #51 (phải giữ `O(số lời giải · n^2)` ký tự output).

## 17. Bảng so sánh các cách

| Cách | Ý tưởng | Mỗi nút tốn | Số nút (`n = 8`) | Dễ viết | Nên dùng khi |
|---|---|---|---|---|---|
| 1. Hoán vị + lọc | sinh `n!` hoán vị, lọc chéo | `O(n^2)` | 40.320 hoán vị | rất dễ | làm baseline để kiểm tra |
| 2. Quét `queens` | đặt từng hàng, quét hậu trước | `O(n^2)` | 2.057 | dễ | mới học backtracking |
| 3. 3 set | nhớ cột, `r-c`, `r+c` | `O(n)` | 2.057 | dễ | phỏng vấn, cân bằng rõ ràng/nhanh |
| 4. Bitmask | 3 số nguyên, dịch chéo | chỉ cột an toàn | 2.057 | trung bình | muốn tối ưu thật sự |
| 5. Bitmask + đối xứng | nửa trái × 2 (+ cột giữa) | như 4 | 1.029 | trung bình | nộp bài, gây ấn tượng |

## 18. Lỗi thường gặp

### Lỗi 1: Quên "bỏ chọn" khi dùng set

```python
cols.add(c); diag1.add(r - c); diag2.add(r + c)
total += backtrack(r + 1)
# quên: cols.remove(c); diag1.remove(r - c); diag2.remove(r + c)
```

Cột và chéo đã thử vẫn bị đánh dấu "chiếm" mãi mãi, nên các nhánh sau bị chặn oan → đếm thiếu.

In [25]:
def buggy_no_undo(n):
    cols, diag1, diag2 = set(), set(), set()

    def backtrack(r):
        if r == n:
            return 1
        total = 0
        for c in range(n):
            if c in cols or (r - c) in diag1 or (r + c) in diag2:
                continue
            cols.add(c); diag1.add(r - c); diag2.add(r + c)
            total += backtrack(r + 1)
        return total

    return backtrack(0)

for n in [4, 5, 6, 8]:
    print(f"n = {n}: code lỗi = {buggy_no_undo(n)}, đúng = {count_backtrack_sets(n)[0]}")

n = 4: code lỗi = 0, đúng = 2
n = 5: code lỗi = 1, đúng = 10
n = 6: code lỗi = 0, đúng = 4
n = 8: code lỗi = 0, đúng = 92


**Sửa**: luôn có cặp đối xứng "thêm trước khi đi sâu — xóa sau khi quay về". Mẹo: viết dòng `remove` ngay khi vừa viết dòng `add`.

### Lỗi 2: Dùng `abs(r - c)` làm mã đường chéo

`abs` gộp hai đường chéo khác nhau làm một: ô `(0, 1)` có `r - c = -1`, ô `(1, 0)` có `r - c = 1`, chúng **không** cùng chéo "\\", nhưng `abs` cho cả hai bằng `1` → chặn oan.

In [26]:
def buggy_abs_diag(n):
    cols, diag1, diag2 = set(), set(), set()

    def backtrack(r):
        if r == n:
            return 1
        total = 0
        for c in range(n):
            if c in cols or abs(r - c) in diag1 or (r + c) in diag2:
                continue
            cols.add(c); diag1.add(abs(r - c)); diag2.add(r + c)
            total += backtrack(r + 1)
            cols.remove(c); diag1.remove(abs(r - c)); diag2.remove(r + c)
        return total

    return backtrack(0)

for n in [4, 5, 6, 8]:
    print(f"n = {n}: code lỗi = {buggy_abs_diag(n)}, đúng = {count_backtrack_sets(n)[0]}")

n = 4: code lỗi = 0, đúng = 2
n = 5: code lỗi = 4, đúng = 10
n = 6: code lỗi = 0, đúng = 4
n = 8: code lỗi = 0, đúng = 92


**Sửa**: dùng đúng `r - c` (có thể âm, `set` vẫn lưu được số âm). Nếu dùng mảng thay set thì dịch chỉ số: `r - c + n - 1` để nằm trong `0 .. 2n-2`.

### Lỗi 3: Quên `& full`

- Viết `available = ~(cols | d1 | d2)` → ra **số âm** có vô hạn bit 1 (mục 10) → `while available` **không bao giờ dừng**. Đừng chạy thử đoạn này!
- Quên `& full` sau `d1 << 1` thì không sai kết quả (vì `available` đã `& full`), nhưng số cứ phình to; thói quen tốt là luôn cắt.

### Lỗi 4: Nhân đôi cả cột giữa khi `n` lẻ

In [27]:
def buggy_symmetry(n):
    by_col = count_by_first_column(n)
    half = (n + 1) // 2                     # SAI: với n lẻ, lấy luôn cột giữa
    return 2 * sum(by_col[:half])

for n in [4, 5, 7, 9]:
    print(f"n = {n}: code lỗi = {buggy_symmetry(n)}, đúng = {count_bitmask_symmetry(n)[0]}")

n = 4: code lỗi = 2, đúng = 2
n = 5: code lỗi = 12, đúng = 10
n = 7: code lỗi = 46, đúng = 40
n = 9: code lỗi = 406, đúng = 352


Với `n` chẵn thì vô tình đúng, nên test chỉ `n = 4, 8` sẽ **không bắt được lỗi** — luôn test cả `n` lẻ. **Sửa**: nửa trái là `range(n // 2)` nhân 2, cột giữa `n // 2` cộng riêng một lần khi `n % 2 == 1`.

### Lỗi 5: Dịch hai chéo cùng chiều, hoặc đếm bằng biến toàn cục không reset

- Nếu viết `(d2 | bit) << 1` cho cả chéo "/" thì vết tấn công đi sai hướng → kết quả sai. Nhớ: "\\" đi xuống **sang cột lớn** (`<<`), "/" đi xuống **sang cột nhỏ** (`>>`).
- Nếu đếm bằng `self.count += 1` mà quên gán lại `self.count = 0` ở đầu `totalNQueens`, gọi hàm hai lần trên cùng một object sẽ cộng dồn kết quả lần trước. Kiểu "hàm trả về số lời giải của cây con" như notebook này tránh được hoàn toàn lỗi đó.

## 19. Mẹo nhận dạng pattern

- Đề nói **"đặt / chọn sao cho không vi phạm ràng buộc"**, input nhỏ (`n <= 9`, `n <= 15`...) → nghĩ ngay tới **backtracking**.
- Đề hỏi **"bao nhiêu cách"** thay vì "liệt kê" → cho hàm đệ quy **trả về số đếm**, bỏ hết phần lưu/copy lời giải.
- Ràng buộc kiểu "mỗi hàng/cột/nhóm tối đa một" → tìm **mã số** cho nhóm (`r - c`, `r + c`, `(r // 3, c // 3)` trong Sudoku) rồi lưu vào `set`.
- Tập hợp con của **ít phần tử** (≤ 20–30) → nén vào **bitmask**; nhớ bộ đôi `x & -x` / `x ^= bit` để duyệt bit 1.
- Bài toán có **đối xứng** (lật, xoay) mà đề đếm cả các bản đối xứng → duyệt một phần rồi nhân lên, cẩn thận phần tử tự đối xứng.
- Ghi chú vui: vì `n <= 9`, một số người nộp bảng tra `[1, 0, 0, 2, 10, 4, 40, 92, 352][n - 1]`. Đúng, nhưng không học được gì — hãy dùng nó để **kiểm tra** code của mình thôi.

## 20. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| #51 N-Queens | cùng cây tìm kiếm, nhưng phải sinh bàn cờ dạng chuỗi |
| #46 Permutations | backtracking sinh hoán vị (chính là cách 1 viết tay) |
| #37 Sudoku Solver | backtracking + 3 nhóm ràng buộc hàng/cột/ô 3×3, có thể dùng bitmask |
| #526 Beautiful Arrangement | đếm hoán vị thỏa điều kiện, rất hợp backtracking + bitmask |
| #980 Unique Paths III | đếm đường đi bằng backtracking, có bước bỏ chọn |
| Biến thể tự chế | đếm khi có **ô cấm**; đếm lời giải **không tính xoay/lật** (bài tập 5, 6 dưới đây) |

## 21. Bài tập

### Bài 1 — Đoán output (phép bit)

Không chạy code, hãy đoán 5 giá trị sau (viết dạng nhị phân 5 bit), rồi xem đáp án cuối notebook:

```python
x = 0b10110
full = 0b11111
print(format(x & -x, "05b"))
print(format(x & (x - 1), "05b"))
print(format(x >> 1, "05b"))
print(format((x << 1) & full, "05b"))
print(format(full & ~x, "05b"))
```

### Bài 2 — Tính tay bitmask

Bàn `n = 5` (`full = 11111`). Hàng 0 đặt hậu ở **cột 0**, hàng 1 đặt hậu ở **cột 2**. Hãy tính bằng tay `cols`, `d1`, `d2` **cho hàng 2** (dạng nhị phân 5 bit) và `available`. Hàng 2 còn đặt được ở cột nào? Kiểm tra lại bằng cách vẽ bàn cờ và dùng `r - c`, `r + c`.

Gợi ý điền vào hàm (trả về tuple 4 số nguyên `(cols, d1, d2, available)`):

In [28]:
def exercise_hand_mask():
    # TODO: thay các số 0 bằng kết quả bạn tính tay, ví dụ 0b00101
    cols = 0
    d1 = 0
    d2 = 0
    available = 0
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_hand_mask() == answer_hand_mask()

### Bài 3 — Điền TODO cho cách 3 (3 set)

In [29]:
def exercise_count_sets(n: int) -> int:
    cols, diag1, diag2 = set(), set(), set()

    def backtrack(r):
        # TODO 1: nếu r == n thì trả về 1
        # TODO 2: total = 0, thử từng cột c
        # TODO 3: bỏ qua c nếu c, r - c hoặc r + c đã bị chiếm
        # TODO 4: chọn (add vào 3 set) -> total += backtrack(r + 1) -> bỏ chọn (remove)
        # TODO 5: trả về total
        pass

    return backtrack(0)

# Tự làm xong rồi mới mở test:
# assert [exercise_count_sets(n) for n in range(1, 10)] == [1, 0, 0, 2, 10, 4, 40, 92, 352]

### Bài 4 — Tự viết lại cách 4 (bitmask) không nhìn code

Yêu cầu: dùng `full`, `available = full & ~(cols | d1 | d2)`, `bit = available & -available`. Không dùng `set`, không dùng mảng `queens`.

In [30]:
def exercise_count_bitmask(n: int) -> int:
    # TODO: viết hàm go(cols, d1, d2) trả về số lời giải
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert [exercise_count_bitmask(n) for n in range(1, 10)] == [1, 0, 0, 2, 10, 4, 40, 92, 352]

### Bài 5 — Biến thể: bàn cờ có ô cấm

Cho `n` và danh sách `blocked` các ô `(r, c)` **không được đặt hậu** (hậu vẫn tấn công xuyên qua ô cấm như bình thường). Đếm số lời giải.

Ví dụ: `n = 4, blocked = [(0, 1)]` → `1` (lời giải A dùng ô `(0, 1)` nên bị loại, còn lời giải B).

Gợi ý nhỏ: chuẩn bị mảng `blocked_mask[r]` là bitmask các cột cấm của hàng `r`.

In [31]:
def exercise_count_with_blocked(n: int, blocked: list) -> int:
    # TODO 1: tạo blocked_mask = [0] * n và bật bit cho từng ô cấm
    # TODO 2: backtracking bitmask, nhưng cần biết đang ở hàng nào để trừ blocked_mask[row]
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_count_with_blocked(4, []) == 2
# assert exercise_count_with_blocked(4, [(0, 1)]) == 1
# assert exercise_count_with_blocked(4, [(0, 1), (0, 2)]) == 0
# assert exercise_count_with_blocked(8, [(0, 0)]) == 88

### Bài 6 — Mini challenge: đếm lời giải "thật sự khác nhau"

Coi hai lời giải là **một** nếu xoay (90°, 180°, 270°) hoặc lật gương bàn cờ thì trùng nhau (8 phép đối xứng của hình vuông). Đếm số lời giải cơ bản. Kết quả đã biết: `n = 4 → 1`, `n = 5 → 2`, `n = 6 → 1`, `n = 8 → 12`.

Bài này buộc bạn **phải sinh lời giải** (như bài #51) — đúng là lúc thấy "chỉ đếm" của #52 dễ hơn bao nhiêu.

In [32]:
def exercise_count_unique(n: int) -> int:
    # TODO 1: sinh tất cả lời giải dạng tuple queens (queens[r] = c)
    # TODO 2: với mỗi lời giải, tạo 8 biến thể bằng xoay/lật, đưa về dạng chuẩn (vd lấy min)
    # TODO 3: đếm số dạng chuẩn khác nhau bằng set
    pass

# Tự làm xong rồi mới mở test:
# assert [exercise_count_unique(n) for n in [4, 5, 6, 8]] == [1, 2, 1, 12]

## 22. Gợi ý theo mức

**Bài 1**
- Mức 1: `x & -x` giữ bit 1 thấp nhất; `x & (x - 1)` xóa bit đó.
- Mức 2: `x = 10110`, bit 1 thấp nhất ở vị trí 1.
- Mức 3: `full & ~x` là đảo đúng 5 bit của `x`.

**Bài 2**
- Mức 1: sau mỗi hàng, `d1` dịch trái, `d2` dịch phải, và đều OR thêm `bit` của hậu vừa đặt **trước khi dịch**.
- Mức 2: sau hàng 0: `cols = 00001`, `d1 = 00010`, `d2 = 00000`.
- Mức 3: hàng 1 đặt `bit = 00100`; `d1 = (00010 | 00100) << 1`, `d2 = (00000 | 00100) >> 1`.

**Bài 3**
- Mức 1: chép nhịp "chọn → đi sâu → bỏ chọn".
- Mức 2: điều kiện bỏ qua: `c in cols or (r - c) in diag1 or (r + c) in diag2`.
- Mức 3: `remove` phải đúng 3 giá trị đã `add`.

**Bài 4**
- Mức 1: dừng khi `cols == full`.
- Mức 2: vòng `while available:` lấy `bit`, xóa `bit`, gọi đệ quy.
- Mức 3: `go(cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)`.

**Bài 5**
- Mức 1: ô cấm giống một ô "bị tấn công" nhưng chỉ ở đúng hàng của nó.
- Mức 2: thêm tham số `row` vào `go`, hoặc suy ra `row` bằng `bin(cols).count("1")`.
- Mức 3: `available = full & ~(cols | d1 | d2 | blocked_mask[row])`.

**Bài 6**
- Mức 1: đổi lời giải thành tập ô `{(r, c)}`, áp phép biến đổi tọa độ, rồi đổi lại thành tuple.
- Mức 2: xoay 90°: `(r, c) -> (c, n-1-r)`; lật trái–phải: `(r, c) -> (r, n-1-c)`. Kết hợp 4 lần xoay × (lật / không lật) = 8 phép.
- Mức 3: đổi tập ô về tuple: `tuple(c for r, c in sorted(cells))`; dạng chuẩn = `min` của 8 tuple.

## 23. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

Giải thích Bài 2 (code kiểm chứng ở cell ngay dưới): `cols = 00101` (cột 0, 2). Chéo "\\": hậu `(0,0)` chạm `(2,2)`, hậu `(1,2)` chạm `(2,3)` → `d1 = 01100`. Chéo "/": hậu `(1,2)` chạm `(2,1)`, hậu `(0,0)` chạm `(2,-2)` đã ra ngoài → `d2 = 00010`. Bị chặn `01111` → `available = 10000`: **chỉ còn cột 4**. Kiểm tra bằng mã: `(2,4)` có `r - c = -2` (khác `0`, `-1`) và `r + c = 6` (khác `0`, `3`) → an toàn.

In [33]:
# ---------- Bài 1 ----------
x = 0b10110
full5 = 0b11111
answer_bits = [
    format(x & -x, "05b"),
    format(x & (x - 1), "05b"),
    format(x >> 1, "05b"),
    format((x << 1) & full5, "05b"),
    format(full5 & ~x, "05b"),
]
print("Bài 1:", answer_bits)
assert answer_bits == ["00010", "10100", "01011", "01100", "01001"]


# ---------- Bài 2 ----------
def answer_hand_mask():
    n = 5
    full = (1 << n) - 1
    cols = d1 = d2 = 0
    for c in [0, 2]:                       # hàng 0 cột 0, hàng 1 cột 2
        bit = 1 << c
        cols, d1, d2 = cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1
    available = full & ~(cols | d1 | d2)
    return cols, d1, d2, available

cols, d1, d2, available = answer_hand_mask()
print("Bài 2: cols =", format(cols, "05b"), "d1 =", format(d1, "05b"),
      "d2 =", format(d2, "05b"), "available =", format(available, "05b"))
assert answer_hand_mask() == (0b00101, 0b01100, 0b00010, 0b10000)   # chỉ còn cột 4

Bài 1: ['00010', '10100', '01011', '01100', '01001']
Bài 2: cols = 00101 d1 = 01100 d2 = 00010 available = 10000


In [34]:
# ---------- Bài 3 ----------
def answer_count_sets(n: int) -> int:
    cols, diag1, diag2 = set(), set(), set()

    def backtrack(r):
        if r == n:
            return 1
        total = 0
        for c in range(n):
            if c in cols or (r - c) in diag1 or (r + c) in diag2:
                continue
            cols.add(c); diag1.add(r - c); diag2.add(r + c)
            total += backtrack(r + 1)
            cols.remove(c); diag1.remove(r - c); diag2.remove(r + c)
        return total

    return backtrack(0)


# ---------- Bài 4 ----------
def answer_count_bitmask(n: int) -> int:
    full = (1 << n) - 1

    def go(cols, d1, d2):
        if cols == full:
            return 1
        total = 0
        available = full & ~(cols | d1 | d2)
        while available:
            bit = available & -available
            available ^= bit
            total += go(cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)
        return total

    return go(0, 0, 0)


assert [answer_count_sets(n) for n in range(1, 10)] == [1, 0, 0, 2, 10, 4, 40, 92, 352]
assert [answer_count_bitmask(n) for n in range(1, 10)] == [1, 0, 0, 2, 10, 4, 40, 92, 352]
print("Bài 3, 4: PASS")


# ---------- Bài 5 ----------
def answer_count_with_blocked(n: int, blocked: list) -> int:
    full = (1 << n) - 1
    blocked_mask = [0] * n
    for r, c in blocked:
        blocked_mask[r] |= 1 << c

    def go(row, cols, d1, d2):
        if row == n:
            return 1
        total = 0
        available = full & ~(cols | d1 | d2 | blocked_mask[row])
        while available:
            bit = available & -available
            available ^= bit
            total += go(row + 1, cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)
        return total

    return go(0, 0, 0, 0)


assert answer_count_with_blocked(4, []) == 2
assert answer_count_with_blocked(4, [(0, 1)]) == 1
assert answer_count_with_blocked(4, [(0, 1), (0, 2)]) == 0
assert answer_count_with_blocked(8, [(0, 0)]) == 88     # 92 - 4 lời giải có hậu ở góc (0, 0)
assert all(answer_count_with_blocked(n, []) == count_bitmask(n)[0] for n in range(1, 10))

# Đối chiếu với brute force trên ô cấm ngẫu nhiên (seed cố định)
def brute_with_blocked(n, blocked):
    bset = set(blocked)
    return sum(1 for p in permutations(range(n))
               if no_diagonal_attack(p) and all((r, p[r]) not in bset for r in range(n)))

random.seed(2024)
for _ in range(30):
    n = random.randint(1, 7)
    blocked = [(random.randrange(n), random.randrange(n)) for _ in range(random.randint(0, 4))]
    assert answer_count_with_blocked(n, blocked) == brute_with_blocked(n, blocked), (n, blocked)
print("Bài 5: PASS")

Bài 3, 4: PASS


Bài 5: PASS


In [35]:
# ---------- Bài 6 ----------
def all_solutions(n):
    result = []
    queens = []
    cols, diag1, diag2 = set(), set(), set()

    def backtrack(r):
        if r == n:
            result.append(tuple(queens))
            return
        for c in range(n):
            if c in cols or (r - c) in diag1 or (r + c) in diag2:
                continue
            queens.append(c); cols.add(c); diag1.add(r - c); diag2.add(r + c)
            backtrack(r + 1)
            queens.pop(); cols.remove(c); diag1.remove(r - c); diag2.remove(r + c)

    backtrack(0)
    return result


def answer_count_unique(n: int) -> int:
    def to_tuple(cells):
        return tuple(c for r, c in sorted(cells))

    canon = set()
    for sol in all_solutions(n):
        cells = [(r, c) for r, c in enumerate(sol)]
        variants = []
        for _ in range(4):
            cells = [(c, n - 1 - r) for r, c in cells]                 # xoay 90 độ
            variants.append(to_tuple(cells))
            variants.append(to_tuple([(r, n - 1 - c) for r, c in cells]))  # lật gương
        canon.add(min(variants))
    return len(canon)


print("Bài 6:", {n: answer_count_unique(n) for n in range(1, 10)})
assert [answer_count_unique(n) for n in [4, 5, 6, 8]] == [1, 2, 1, 12]
assert all(len(all_solutions(n)) == count_bitmask(n)[0] for n in range(1, 10))
print("Bài 6: PASS")

Bài 6: {1: 1, 2: 0, 3: 0, 4: 1, 5: 2, 6: 1, 7: 6, 8: 12, 9: 46}
Bài 6: PASS


## 24. Template nên nhớ

**Backtracking đếm (3 set)** — dễ nhớ nhất:

```python
def backtrack(r):
    if r == n:
        return 1
    total = 0
    for c in range(n):
        if c in cols or r - c in d1 or r + c in d2:
            continue
        # chọn -> đi sâu -> bỏ chọn
        cols.add(c); d1.add(r - c); d2.add(r + c)
        total += backtrack(r + 1)
        cols.remove(c); d1.remove(r - c); d2.remove(r + c)
    return total
```

**Bitmask** — 4 dòng lõi:

```python
available = full & ~(cols | d1 | d2)
while available:
    bit = available & -available; available ^= bit
    total += go(cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)
```

**Đối xứng**: `2 × (các cột 0 .. n//2 - 1)` + `(cột n//2 nếu n lẻ)`.

Câu thần chú: *mỗi hàng một hậu · chéo "\\" là `r - c`, chéo "/" là `r + c` · chỉ đếm thì trả về số, đừng dựng bàn cờ.*

## 25. Checklist trước khi submit

- Hàm đệ quy có **trả về** số lời giải (không quên `return total`)?
- Có đủ cặp **chọn / bỏ chọn** (nếu dùng set hoặc mảng)?
- Mã chéo đúng là `r - c` và `r + c`, **không** dùng `abs(r - c)`?
- Bitmask: có `& full` khi tính `available`? Chéo "\\" dịch `<<`, chéo "/" dịch `>>`?
- Đối xứng: nửa trái là `range(n // 2)`, cột giữa chỉ cộng **một lần** khi `n` lẻ?
- Đã test `n = 1` (→ 1), `n = 2, 3` (→ 0), `n = 4` (→ 2), một `n` lẻ như `5` (→ 10) và `n = 9` (→ 352)?
- Nếu dùng biến đếm kiểu `self.count`, đã reset về 0 ở đầu mỗi lần gọi chưa?